# Multi-Step Agent Repair: Which Restart Strategy Wins?

**AAAI'27 Experiment Pipeline on Google Colab**

This notebook runs the full 7-stage pipeline on a Colab GPU (T4/L4).
Results are saved to Google Drive so they persist across sessions.

| Stage | What it does | ~Time (T4, 500 Qs) |
|---|---|---|
| 0 | Download HotpotQA, create folders | 1 min |
| 1 | ReAct trajectory generation | 1-2 h |
| 2 | Step-level uncertainty scoring | 1-2 h |
| 3 | 72B/32B judge error annotation | 1-2 h |
| 4 | Localization scoring | < 1 min |
| 5 | 18 repair strategies | 2-4 h |
| 6 | Tables, stats, figures | < 1 min |

**Every stage is resumable** — if Colab disconnects, just re-run the cell and it picks up where it left off.

## 0. Setup: Mount Drive, Clone Repo, Install Dependencies

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab.yaml'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kishormorol/agent-repair.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install dependencies (pinned versions for vLLM compatibility)
!pip install -q -r requirements.txt 2>&1 | tail -5
print('\n--- Installation complete ---')

In [ ]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Stage 0: Download HotpotQA + Create Folders

In [ ]:
!python scripts/run_setup.py --config {CONFIG}

## Stage 1: Generate ReAct Trajectories

Runs the ReAct agent on 500 HotpotQA questions (batched). Failed trajectories are saved for repair.

In [ ]:
!python scripts/run_generate.py --config {CONFIG}

## Stage 2: Step-Level Uncertainty

Computes 5 uncertainty metrics (token entropy, perplexity, max-token-prob, self-consistency, verbalized confidence) for every step.

In [ ]:
!python scripts/run_uncertainty.py --config {CONFIG}

## Stage 3: Error Annotation (Judge)

Uses a 32B judge model (fallback for Colab VRAM) to mark the true broken step in each failed trajectory.

In [ ]:
!python scripts/run_annotate.py --config {CONFIG}

## Stage 4: Localization Scoring

Tests whether uncertainty metrics correctly identify the broken step (5 metrics × 3 rules). CPU-only, fast.

In [ ]:
!python scripts/run_localize.py --config {CONFIG}

## Stage 5: Repair (18 Strategies)

Runs all 18 repair strategies (3 baselines + 5 metrics × 3 rules) on every failure, with deduplication. This is the longest stage.

In [ ]:
!python scripts/run_repair.py --config {CONFIG}

## Stage 6: Evaluation — Tables, Stats, Figures

Generates result tables, significance tests (McNemar + Holm), and figures.

In [ ]:
!python scripts/run_eval.py --config {CONFIG}

## View Results

In [ ]:
from src.utils import load_config
cfg = load_config(CONFIG)

# Print summary
summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())
else:
    print('Summary not yet generated. Run Stage 6 first.')

In [ ]:
import pandas as pd

results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)
else:
    print('Results table not yet generated. Run Stage 6 first.')

In [ ]:
# Display generated figures
import glob
from IPython.display import display, Image

fig_dir = cfg.path('figures')
figs = sorted(glob.glob(os.path.join(fig_dir, '*.png')))
if figs:
    for f in figs:
        print(f'\n--- {os.path.basename(f)} ---')
        display(Image(filename=f, width=800))
else:
    print('No figures yet. Run Stage 6 first.')

## (Optional) Smoke Test

Run all stages on just 20 questions first to verify everything works.

In [ ]:
# Uncomment to run smoke test (20 questions, ~10-15 min total)
# for stage in ['run_setup', 'run_generate', 'run_uncertainty', 'run_annotate',
#               'run_localize', 'run_repair', 'run_eval']:
#     print(f'\n{"="*60}\n  {stage}\n{"="*60}')
#     !python scripts/{stage}.py --config {CONFIG} --limit 20